In [1]:
!git clone -b mammography https://github.com/emina-demirovic/Diffusion-Model-for-Anomaly-Detection-in-Mammography-Records.git

Cloning into 'Diffusion-Model-for-Anomaly-Detection-in-Mammography-Records'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 172 (delta 48), reused 172 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 4.45 MiB | 17.45 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [2]:
%cd Diffusion-Model-for-Anomaly-Detection-in-Mammography-Records

/content/Diffusion-Model-for-Anomaly-Detection-in-Mammography-Records


In [3]:
!git branch

* mammography


In [ ]:
!python --version
!pip install -r requirements.txt
!pip install -e .
!pip install omegaconf

In [ ]:
import ddpm-for-anomaly-detection
print("Project import OK")

In [ ]:
from pathlib import Path

# The INBreast dataset is not distributed with this repository.
# Access should be requested directly from the original dataset authors.
INBREAST_DATASET_SOURCE = "url_to_INBreast_dataset"

# After obtaining authorized access, upload the archive to this location.
INBREAST_ARCHIVE = Path("/content/INBreast.zip")

if not INBREAST_ARCHIVE.exists():
    raise FileNotFoundError(
        "INBreast.zip was not found. Obtain the dataset from its original "
        "authors and upload the authorized archive to /content/INBreast.zip."
    )


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1UBWn7IcxtlpaD9kQ5fA97X6meHc9sKNi
From (redirected): https://drive.google.com/uc?id=1UBWn7IcxtlpaD9kQ5fA97X6meHc9sKNi&confirm=t&uuid=2c5d5c5a-eb93-41d8-af1d-c0d2a4008786
To: /content/INBreast.zip
100% 2.29G/2.29G [00:50<00:00, 45.6MB/s]


In [7]:
!unzip -q /content/INBreast.zip -d /content/dataset_raw/INBreast

In [21]:
!find /content/dataset_raw/INBreast -maxdepth 2 -type d

/content/dataset_raw/INBreast
/content/dataset_raw/INBreast/AllXML
/content/dataset_raw/INBreast/extras
/content/dataset_raw/INBreast/extras/MassSegmentationMasks
/content/dataset_raw/INBreast/extras/CalcificationSegmentationMasks
/content/dataset_raw/INBreast/extras/extrasUpdatedVer
/content/dataset_raw/INBreast/PectoralMuscle
/content/dataset_raw/INBreast/PectoralMuscle/Pectoral Muscle ROI
/content/dataset_raw/INBreast/PectoralMuscle/Pectoral Muscle XML
/content/dataset_raw/INBreast/MedicalReports
/content/dataset_raw/INBreast/AllROI
/content/dataset_raw/INBreast/AllDICOMs


In [ ]:
!pip install pydicom

In [20]:
!mv /content/dataset_raw/INBreast/INBreast/* /content/dataset_raw/INBreast/
!rmdir /content/dataset_raw/INBreast/INBreast

In [23]:
!python scripts/prepare_dataset.py

=== Preparing INBreast dataset ===
Total DICOMs: 410
Mass cases: 107
Healthy candidates: 303
Train healthy: 243
Test healthy: 60

Saving train healthy: 100% 243/243 [00:57<00:00,  4.26it/s]
Saving test healthy: 100% 60/60 [00:16<00:00,  3.66it/s]
Saving test mass + masks: 100% 107/107 [00:53<00:00,  2.01it/s]

Done.
Saved dataset to: /content/datasets/INBreast


In [24]:
!python scripts/check_dataset.py

=== Checking prepared INBreast dataset ===
train/healthy: 243 files, sizes: {(256, 256)}
test/healthy: 60 files, sizes: {(256, 256)}
test/mass: 107 files, sizes: {(256, 256)}
test/masks: 107 files, sizes: {(256, 256)}

Mass images without mask: 0
Masks without mass image: 0
Blank masks after resize: 0


In [ ]:
!pip install pie-torch
!pip install torchgeometry

In [ ]:
config_path = "ddpm-for-anomaly-detection/configs/inbreast_debug.yaml"

with open(config_path, "r") as f:
    text = f.read()

text = text.replace('report_to: tensorboard', 'report_to: null')

with open(config_path, "w") as f:
    f.write(text)

print("Updated report_to to null")

Updated report_to to null


In [ ]:
!git restore ddpm-for-anomaly-detection/configs/inbreast_debug.yaml
!rm -rf experiments
!git pull origin mammography

In [ ]:
!git pull origin mammography
!rm -rf experiments

In [ ]:
!python ddpm-for-anomaly-detection/ours_trainer.py \
  --h_config=./ddpm-for-anomaly-detection/configs/inbreast_debug.yaml \
  --modality=INBREAST \
  --datasets_dir=/content/datasets \
  --image_size=256 \
  --center=True \
  --normal_split=0.8 \
  --anomal_split=0.99 \
  --num_images_log=1 \
  --val_steps=10 \
  --log_frequency=1

In [65]:
!find experiments -maxdepth 6 -type d | sort

experiments
experiments/inbreast_debug
experiments/inbreast_debug/INBREAST_fold0
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet_ema
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/unet
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/unet_ema


In [66]:
!find experiments -maxdepth 6 -type f | sort | head -100

experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/custom_checkpoint_0.pkl
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/optimizer.bin
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/random_states_0.pkl
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/scaler.pt
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/scheduler.bin
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet/config.json
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet/diffusion_pytorch_model.safetensors
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet_ema/config.json
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet_ema/diffusion_pytorch_model.safetensors
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/custom_checkpoint_0.pkl
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/optimizer.bin
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/random_states_0.pkl
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/scaler.pt
ex

In [ ]:
!git pull origin mammography

In [ ]:
!python ddpm-for-anomaly-detection/ours_trainer.py \
  --h_config=./ddpm-for-anomaly-detection/configs/inbreast_debug.yaml \
  --modality=INBREAST \
  --datasets_dir=/content/datasets \
  --image_size=256 \
  --center=True \
  --normal_split=0.8 \
  --anomal_split=0.99 \
  --eval=True \
  --num_images_log=1 \
  --val_steps=1000 \
  --log_frequency=1

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using cuda.
INFO:__main__:[RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

Resuming from checkpoint experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
INFO:accelerate.accelerator:[RANK 0] Loading states from experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:282: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d.UNet2DModel'>.load_config(...) followed by <class 'diffusers.models.unets.une

In [97]:
!find experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10 -maxdepth 3 -type f | sort

experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/anomaly_maps.pt
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/anomaly_scores.pt
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/labels.pt


In [98]:
import torch
from pathlib import Path

eval_dir = Path("experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10")

for name in ["anomaly_maps.pt", "anomaly_scores.pt", "labels.pt"]:
    path = eval_dir / name
    obj = torch.load(path, map_location="cpu")

    print("\n", name)
    print("type:", type(obj))

    if hasattr(obj, "shape"):
        print("shape:", obj.shape)
        print("min:", obj.min().item() if obj.numel() > 0 else None)
        print("max:", obj.max().item() if obj.numel() > 0 else None)
    elif isinstance(obj, list):
        print("len:", len(obj))
        if len(obj) > 0 and hasattr(obj[0], "shape"):
            print("first shape:", obj[0].shape)
    else:
        print(obj)


 anomaly_maps.pt
type: <class 'torch.Tensor'>
shape: torch.Size([10, 1, 256, 256])
min: 0.00030100345611572266
max: 0.11982971429824829

 anomaly_scores.pt
type: <class 'torch.Tensor'>
shape: torch.Size([10])
min: 0.006707237102091312
max: 0.014046130701899529

 labels.pt
type: <class 'torch.Tensor'>
shape: torch.Size([10])
min: 0
max: 1


In [ ]:
!zip -r inbreast_initial_results.zip experiments/inbreast_debug/INBREAST_fold0

In [105]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [106]:
!mkdir -p /content/drive/MyDrive/inbreast_diffusion_results
!cp inbreast_initial_results.zip /content/drive/MyDrive/inbreast_diffusion_results/

In [ ]:
%cd /content/Diffusion-Model-for-Anomaly-Detection-in-Mammography-Records
!git pull origin mammography

In [ ]:
!python ddpm-for-anomaly-detection/ours_trainer.py \
  --h_config=./ddpm-for-anomaly-detection/configs/inbreast_debug.yaml \
  --modality=INBREAST \
  --datasets_dir=/content/datasets \
  --image_size=256 \
  --center=True \
  --normal_split=0.8 \
  --anomal_split=0.99 \
  --eval=True \
  --num_images_log=1 \
  --val_steps=1000 \
  --log_frequency=1

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using cuda.
INFO:__main__:[RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

Resuming from checkpoint experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
INFO:accelerate.accelerator:[RANK 0] Loading states from experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:282: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d.UNet2DModel'>.load_config(...) followed by <class 'diffusers.models.unets.une

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from argparse import Namespace

from ddpm-for-anomaly-detection.data.inbreast_datasets import get_datasets_inbreast

eval_dir = Path("experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10")
maps = torch.load(eval_dir / "anomaly_maps.pt", map_location="cpu")
scores = torch.load(eval_dir / "anomaly_scores.pt", map_location="cpu")
labels = torch.load(eval_dir / "labels.pt", map_location="cpu")

cfg = Namespace(
    datasets_dir="/content/datasets",
    image_size=256,
    center=True,
    percentage=100,
    seed=10,
    batch_size=1,
    normal_split=0.8,
    anomal_split=0.99,
    shuffle=False,
    aug_fn=None
)

test_dset, _ = get_datasets_inbreast(cfg, False)

all_labels = test_dset.labels
anomalous_indices = [i for i, y in enumerate(all_labels) if y == 1]
healthy_indices = [i for i, y in enumerate(all_labels) if y == 0]

def pick_evenly(indices, k):
    if len(indices) <= k:
        return indices
    pos = np.linspace(0, len(indices) - 1, k, dtype=int)
    return [indices[p] for p in pos]

selected_anomalous = pick_evenly(anomalous_indices, 5)
selected_healthy = pick_evenly(healthy_indices, 5)
mini_indices = selected_anomalous + selected_healthy

vis_dir = eval_dir / "visualizations"
vis_dir.mkdir(exist_ok=True)

for row, ds_idx in enumerate(mini_indices):
    img, mask = test_dset[ds_idx]
    img = img[0].numpy()
    mask = mask[0].numpy()

    # [-1, 1] -> [0, 1]
    if cfg.center:
        img = img / 2 + 0.5

    anomaly_map = maps[row, 0].numpy()
    score = float(scores[row])
    label = int(labels[row])

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"Original\nlabel={label}")
    axes[0].axis("off")

    axes[1].imshow(anomaly_map, cmap="hot")
    axes[1].set_title(f"Anomaly map\nscore={score:.5f}")
    axes[1].axis("off")

    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Ground-truth mask")
    axes[2].axis("off")

    plt.tight_layout()
    out_path = vis_dir / f"sample_{row:02d}_label_{label}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

print("Saved visualizations to:", vis_dir)

Saved visualizations to: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations


In [115]:
!find experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations -type f | sort

experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_00_label_1.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_01_label_1.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_02_label_1.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_03_label_1.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_04_label_1.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_05_label_0.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_06_label_0.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_07_label_0.png
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_08_label_0.png
experiments/inbreast_debug/INBREAST_f

In [116]:
!find experiments/inbreast_debug/INBREAST_fold0 -maxdepth 3 -type d | sort

experiments/inbreast_debug/INBREAST_fold0
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet
experiments/inbreast_debug/INBREAST_fold0/checkpoint-10/unet_ema
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/unet
experiments/inbreast_debug/INBREAST_fold0/checkpoint-5/unet_ema
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10
experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations


In [117]:
!zip -r balanced_eval_results.zip experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10

  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/ (stored 0%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/balanced_mini_eval_scores.csv (deflated 41%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/ (stored 0%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_09_label_0.png (deflated 3%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_03_label_1.png (deflated 2%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_00_label_1.png (deflated 2%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_06_label_0.png (deflated 2%)
  adding: experiments/inbreast_debug/INBREAST_fold0/eval_INBREAST_checkpoint-10/visualizations/sample_02_label_1.png (deflated 3%)
  adding: experimen

In [118]:
!cp balanced_eval_results.zip /content/drive/MyDrive/inbreast_diffusion_results/